# W2 — Training (Kaggle GPU)

**Trước khi chạy:** Settings → *Internet: ON*, *Accelerator: **GPU T4 x1** (hoặc P100)*.
**Add Data →** dataset `lits-processed` (từ W1: `images_u8.npy` + `manifest.csv` + `splits/`).

Luồng: verify → train baseline **ResNet-50** (fold 0) → eval patient-level (AUROC + CI + ROC/PR/calibration).
Đổi `--arch convnextv2_nano` để chạy model chính; các backbone khác xem `configs/train/base.yaml`.

In [ ]:
# 1) Thư viện (Kaggle có sẵn torch/pandas/sklearn/matplotlib)
!pip install -q -U timm && pip install -q mlflow albumentations

In [ ]:
# 2) Nạp code + set DATA_ROOT (= dataset lits-processed)
import os
if not os.path.exists('HCC-TACE-Assist'):
    !git clone -q https://github.com/hdtruong802/HCC-TACE-Assist.git
%cd HCC-TACE-Assist
import sys; sys.path.insert(0, '.')
os.environ['DATA_ROOT'] = '/kaggle/input/lits-processed'
print('DATA_ROOT =', os.environ['DATA_ROOT'])
import torch; print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

In [ ]:
# 3) VERIFY: manifest + split + mảng ảnh khớp nhau
import pandas as pd, numpy as np, json
R = os.environ['DATA_ROOT']
df = pd.read_csv(f'{R}/manifest.csv'); arr = np.load(f'{R}/images_u8.npy', mmap_mode='r')
sp = json.load(open(f'{R}/splits/lits_v1.json'))
print('slices:', len(df), '| images:', arr.shape, '| patients:', df.patient_id.nunique())
print('folds:', len(sp['folds']), '| test bn:', len(sp['test']), '| pos_ratio:', round((df.label==1).mean(),3))
assert arr.shape[0] == len(df), 'images_u8.npy KHONG khop manifest!'

In [ ]:
# 4) TRAIN baseline ResNet-50 (fold 0)
!python -m src.training.train --config configs/train/base.yaml \
    --data-root /kaggle/input/lits-processed --arch resnet50 --fold 0

In [ ]:
# 5) EVAL trên validation (patient-level) + curves
!python -m src.evaluation.evaluate --config configs/train/base.yaml \
    --data-root /kaggle/input/lits-processed \
    --ckpt /kaggle/working/outputs/resnet50_fold0/best.ckpt --split val --fold 0
from IPython.display import Image, display
for f in ['roc','pr','confusion','calibration']:
    p = f'/kaggle/working/outputs/resnet50_fold0/eval_val/{f}.png'
    if os.path.exists(p): display(Image(p))

## Bước tiếp
- **Model chính:** đổi cell 4 → `--arch convnextv2_nano` (hoặc `convnextv2_tiny`). So sánh AUROC val.
- **So sánh nhanh (Pha 1):** chạy mỗi arch 1 lần (fold 0) → xếp hạng theo `val_auroc`.
- **Pha 2 (nghiêm ngặt):** 1–2 model top × 5 fold (`--fold 0..4`) × nhiều seed → gộp CI. *(dùng FPT H100 nếu Kaggle hết quota)*.
- **Internal test (1 lần):** `--split test` với ngưỡng đã khóa trên val.
- Lưu `outputs/` (ckpt + metrics + curves); tải `val_metrics.json` về commit để so sánh.